In [5]:
import os
import json
import time
import gzip
import shutil
import subprocess
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
from pathlib import Path
from tqdm import tqdm

# ── directories ────────────────────────────────────────────────────────────────
DATA_DIR    = Path("../vep_data")
PARQUET_DIR = DATA_DIR / "parquet"
VCF_DIR     = DATA_DIR / "vcf"
for d in (PARQUET_DIR, VCF_DIR):
    d.mkdir(parents=True, exist_ok=True)

ENSEMBL_REST = "https://rest.ensembl.org"
ENSEMBL_FTP  = "https://ftp.ensembl.org/pub"

# ── 1. REST API – fetch VEP for a list of variants ────────────────────────────

EXAMPLE_VARIANTS = [
    # HGVS notation (works for SNPs, indels, etc.)
    "9:g.22125504G>C",   # CDKN2A
    "7:g.117548628T>A",  # CFTR
    "17:g.43094692G>A",  # BRCA1
    "13:g.32315474A>T",  # BRCA2
    "12:g.25398284C>A",  # KRAS G12V
    "17:g.7674220C>T",   # TP53
    "3:g.178936091A>G",  # PIK3CA
    "10:g.89692905A>G",  # PTEN
    "7:g.140453136A>T",  # BRAF V600E
    "2:g.29443613C>T",   # ALK
]

con = duckdb.connect()
glob = str(PARQUET_DIR / "**" / "*.parquet")


In [ ]:
q2 = con.execute(f"""
    SELECT chrom, pos, ref, alt, gene_symbol, consequence_terms, impact
    FROM read_parquet('{glob}', hive_partitioning=true)
    WHERE chrom = '2'
        AND pos BETWEEN 43000000 AND 44000000
        AND impact IN ('HIGH', 'MODERATE')
    ORDER BY pos
""").df()

BinderException: Binder Error: Referenced column "gene_symbol" not found in FROM clause!
Candidate bindings: "csq"

LINE 2:     SELECT chrom, pos, ref, alt, gene_symbol, consequence_terms, impact
                                         ^

In [ ]:


duckdb.query("""
    SELECT * FROM read_parquet('vep_data/parquet/**/*.parquet', hive_partitioning=true)
    WHERE pos BETWEEN 1000000 AND 2000000
""")

In [ ]:
"""
Demonstrates common VEP query patterns using DuckDB against Parquet files.
DuckDB reads only the relevant row-groups — no server required.
"""
con = duckdb.connect()
glob = str(parquet_dir / "**" / "*.parquet")

print("\n" + "═" * 60)
print("DuckDB / Parquet queries")
print("═" * 60)

# ── Q1: positional lookup (single variant) ────────────────────
print("\n── Q1: All consequences for a single genomic position ──")
q1 = con.execute(f"""
    SELECT gene_symbol, transcript_id, consequence_terms, impact, hgvsp
    FROM read_parquet('{glob}', hive_partitioning=true)
    WHERE chrom = '17' AND pos = 43094692
    ORDER BY canonical DESC, impact
""").df()
print(q1.to_string(index=False))

# ── Q2: region query ──────────────────────────────────────────
print("\n── Q2: High-impact variants in a genomic window ──")
q2 = con.execute(f"""
    SELECT chrom, pos, ref, alt, gene_symbol, consequence_terms, impact
    FROM read_parquet('{glob}', hive_partitioning=true)
    WHERE chrom = '17'
        AND pos BETWEEN 43000000 AND 44000000
        AND impact IN ('HIGH', 'MODERATE')
    ORDER BY pos
""").df()
print(q2.to_string(index=False))

# ── Q3: filter by gene ────────────────────────────────────────
print("\n── Q3: Canonical transcripts for BRCA1 ──")
q3 = con.execute(f"""
    SELECT pos, ref, alt, consequence_terms, impact, hgvsc, hgvsp,
            sift_prediction, polyphen_prediction
    FROM read_parquet('{glob}', hive_partitioning=true)
    WHERE gene_symbol = 'BRCA1' AND canonical = 1
    ORDER BY pos
""").df()
print(q3.to_string(index=False))

# ── Q4: filter by consequence type ───────────────────────────
print("\n── Q4: All stop_gained / frameshift variants ──")
q4 = con.execute(f"""
    SELECT chrom, pos, gene_symbol, consequence_terms, hgvsp
    FROM read_parquet('{glob}', hive_partitioning=true)
    WHERE consequence_terms LIKE '%stop_gained%'
        OR consequence_terms LIKE '%frameshift%'
""").df()
print(q4.to_string(index=False))

# ── Q5: SIFT / PolyPhen damaging filter ──────────────────────
print("\n── Q5: Predicted deleterious variants (SIFT + PolyPhen) ──")
q5 = con.execute(f"""
    SELECT chrom, pos, gene_symbol, consequence_terms,
            sift_score, sift_prediction,
            polyphen_score, polyphen_prediction
    FROM read_parquet('{glob}', hive_partitioning=true)
    WHERE sift_prediction    LIKE '%deleterious%'
        AND polyphen_prediction LIKE '%damaging%'
        AND canonical = 1
    ORDER BY sift_score ASC
""").df()
print(q5.to_string(index=False))

# ── Q6: aggregate – impact counts per gene ────────────────────
print("\n── Q6: Variant impact counts per gene ──")
q6 = con.execute(f"""
    SELECT gene_symbol,
            COUNT(DISTINCT pos)                                     AS n_variants,
            SUM(CASE WHEN impact='HIGH'     THEN 1 ELSE 0 END)     AS high,
            SUM(CASE WHEN impact='MODERATE' THEN 1 ELSE 0 END)     AS moderate,
            SUM(CASE WHEN impact='LOW'      THEN 1 ELSE 0 END)     AS low
    FROM read_parquet('{glob}', hive_partitioning=true)
    WHERE canonical = 1
    GROUP BY gene_symbol
    ORDER BY high DESC, moderate DESC
""").df()
print(q6.to_string(index=False))

# ── Q7: export filtered subset ────────────────────────────────
print("\n── Q7: Export BRCA1 + BRCA2 HIGH/MODERATE variants to CSV ──")
out_csv = DATA_DIR / "brca_variants.csv"
con.execute(f"""
    COPY (
        SELECT chrom, pos, ref, alt, gene_symbol,
                consequence_terms, impact, hgvsc, hgvsp,
                sift_score, polyphen_score
        FROM read_parquet('{glob}', hive_partitioning=true)
        WHERE gene_symbol IN ('BRCA1', 'BRCA2')
            AND impact IN ('HIGH', 'MODERATE')
            AND canonical = 1
        ORDER BY chrom, pos
    ) TO '{out_csv}' (HEADER, DELIMITER ',')
""")
print(f"    → Saved to {out_csv}")

